# Session 4 — key check (run before class, ~5 min)

Four cells. If the last one prints **READY**, you are done.

Session 4 runs on the **same OpenRouter key as sessions 1, 2 and 3** — provided,
credited, nothing to sign up for. What is new is what this session asks of it: **tool
calling**. A model that answers a normal question perfectly can still refuse to emit a
tool call, and a model that emitted them last month can stop. That is what cell 3
checks, and it is the whole reason this notebook exists.

| what | where | needed? |
|---|---|---|
| **Model key** — OpenRouter | the key handed out in session 1 | yes |
| **Search key** — Tavily | `tavily.com` | **no. Genuinely optional.** |

**About the search key.** You do not need one. The lab's `search_web` tool has a
`cached` mode backed by about fifty recorded searches, and **every part of the lab
completes without a search key**. Your search latencies then come from a recording and
you label them as such in your report. Do not spend time on this before class.

If your key does not answer, or answers without emitting a tool call, email the
instructor **before** the session. A key problem costs you lab time and cannot be fixed
in the first ten minutes of class.

In [ ]:
# Cell 1/4 — install and configure
!pip install -q openai

import json, time
from getpass import getpass
from openai import OpenAI

# Keep in sync with the lab notebook. One endpoint, one key, one model id — and the
# id is the only line that ever has to change. Instructor: confirm it at J-7, and
# confirm it still supports tool calling.
BASE_URL = "https://openrouter.ai/api/v1"
MODEL    = "deepseek/deepseek-chat-v3.1"

print(f"{MODEL} via OpenRouter ({BASE_URL})")

In [ ]:
# Cell 2/4 — does your key answer at all?
client, basic, detail = None, "SKIPPED", "no key entered"

key = getpass("OpenRouter API key: ").strip()
if key:
    client = OpenAI(api_key=key, base_url=BASE_URL, max_retries=5)
    try:
        t0 = time.time()
        r = client.chat.completions.create(
            model=MODEL, temperature=0,
            messages=[{"role": "user", "content": "Reply with exactly the word: ready"}],
        )
        basic = "OK"
        detail = (f"{time.time() - t0:.2f}s, {r.usage.prompt_tokens} in / "
                  f"{r.usage.completion_tokens} out")
    except Exception as e:
        basic, detail = "FAILED", f"{type(e).__name__}: {str(e)[:120]}"

print(f"answers   : {basic:8s} {detail}")

In [ ]:
# Cell 3/4 — THE ONE THAT MATTERS: does it emit a tool call?
# Session 4 is a loop around tool calling. A key that passes cell 2 perfectly and
# fails here is useless for this lab, and finding that out now costs you nothing.

TEST_TOOL = [{"type": "function", "function": {
    "name": "query_orders",
    "description": "Look up Velora orders by month. Returns the matching order rows.",
    "parameters": {"type": "object",
                   "properties": {"month": {"type": "string",
                                            "description": "YYYY-MM, e.g. 2026-03"}},
                   "required": ["month"]}}}]

tools_ok, tools_detail = "SKIPPED", "cell 2 did not produce a working client"

if basic == "OK":
    try:
        r = client.chat.completions.create(
            model=MODEL, temperature=0,
            messages=[{"role": "user",
                       "content": "How many Velora orders were placed in March 2026? "
                                  "Use the tool."}],
            tools=TEST_TOOL, tool_choice="auto")
        m = r.choices[0].message
        if m.tool_calls:
            tc = m.tool_calls[0]
            tools_ok = "OK"
            tools_detail = f"called {tc.function.name}({tc.function.arguments})"
        else:
            tools_ok = "NO TOOL CALL"
            tools_detail = f"answered in prose instead: {(m.content or '')[:70]!r}"
    except Exception as e:
        tools_ok, tools_detail = "FAILED", f"{type(e).__name__}: {str(e)[:120]}"

print(f"tool call : {tools_ok:14s} {tools_detail}")

In [ ]:
# Cell 4/4 — verdict
print()
if tools_ok == "OK":
    print("READY — your key answers AND the model emits tool calls.")
    print("That is everything session 4 needs. See you in class.")
    print()
    print("You do NOT need a search key. The lab runs with SEARCH_MODE = 'cached'.")
elif basic == "OK":
    print("NOT READY — your key answers, but the model did not emit a tool call.")
    print("This is the failure this notebook exists to catch, and it is not your")
    print("fault: a model can lose tool calling from one week to the next. The repair")
    print("is one model id in the lab notebook and it is the instructor's to make —")
    print("email them the exact line printed by cell 3, BEFORE the session.")
else:
    print("NOT READY — your key did not answer.")
    print("Re-read the error in cell 2. The two usual causes: a key pasted with a")
    print("stray space, and a key whose credit has run out. It is the same key as")
    print("sessions 1 to 3, so if it worked last week, suspect the paste first.")
    print("Email the instructor BEFORE the session.")

print()
print("Bring this to class: your key answers =", basic == "OK",
      "/ it emits tool calls =", tools_ok == "OK")